# Train mmBERT mask-targeted classifier — StereoQueerEval Task A

Binary stereotype detection (`yes` / `no`). This notebook clones `src/`, installs the
dependencies, locates the TSVs, then trains and evaluates.

`src/train.py` runs as a plain script, so everything below is a subprocess call —
the module-level `sys.path` tricks that notebooks need do not apply.

In [ ]:
!git clone --depth 1 https://github.com/AmnO-O/Try.git
%cd Try
!pip install -q -r requirements.txt
!ls src

## 1. Put the data where `config.py` expects it

The TSVs are gitignored, so clone alone gives you no data. Two options — pick one:

* **Local**: copy `StereoQueerEval_{EN,IT,NL}_training.tsv` into `<repo>/LGBT/`.
* **Elsewhere**: set `LGBT_DATA_DIR` to whichever directory holds them.

`config.py` also honours `LGBT_WEIGHTS_DIR`, where checkpoints are written.

In [ ]:
import os, glob, sys

# Option A: point at a directory you already have the TSVs in.
os.environ['LGBT_DATA_DIR'] = '/content/data'      # change to your path
os.environ['LGBT_WEIGHTS_DIR'] = '/content/weights'

found = sorted(glob.glob(os.path.join(os.environ['LGBT_DATA_DIR'], 'StereoQueerEval_*_training.tsv')))
print('data dir :', os.environ['LGBT_DATA_DIR'])
print('files    :', [os.path.basename(p) for p in found] or 'NONE FOUND')
assert found, 'No TSVs found. Upload them, or correct LGBT_DATA_DIR.'

## 2. Verify the data loads

Checks row counts, labels and the video-grouped split before any GPU time is spent.

In [ ]:
sys.path.insert(0, os.path.abspath('src'))
import data as data_mod
from config import MAX_LEN

df = data_mod.load_frame(max_len=MAX_LEN)
print('rows:', len(df))
print('labels:', data_mod.label_distribution(df))
print('by language:', df['lang'].value_counts().to_dict())
print('prompt words (max):', int(df['prompt'].str.split().str.len().max()))

tr, va = data_mod.split_by_video(df, 0.1, 42)
print('train/val:', len(tr), '/', len(va))

## 3. Train

Defaults from `config.py`: last 2 encoder blocks unfrozen, `readout_init='random'`,
`max_len=320`, early stopping on validation macro-F1 with `patience=5`.

Run the same command twice, once per readout init, to compare:

    --readout-init random
    --readout-init verbalizer

In [ ]:
!python src/train.py --epochs 10 --batch-size 32 --max-len 320 --unfreeze-layers 2 --readout-init random

## 4. Re-run the same settings with `--readout-init verbalizer`

Both readouts are the same `nn.Linear(768, 2)`; only their starting weights differ.
The verbalizer path copies the pretrained MLM decoder rows for `' no'` and `' yes'`,
so it starts as a real vocabulary readout. Pick the winner by validation macro-F1.

In [ ]:
!python src/train.py --epochs 10 --batch-size 32 --max-len 320 --unfreeze-layers 2 --readout-init verbalizer

## 5. Write predictions

`--readout-init` must match the value used in training: the state dict loads either
way, but a mismatch would silently score with the wrong weights.

In [ ]:
!python src/predict.py --readout-init random --out predictions.csv --device cuda
!head -5 predictions.csv

## 6. Compare the runs

Checkpoints land in `WEIGHTS_DIR`. The best one is `best_mmbert_stereotype_prompt.pt`.

In [ ]:
import os, torch
from IPython.display import display
import pandas as pd

print('checkpoints:')
for f in sorted(os.listdir(os.environ['LGBT_WEIGHTS_DIR'])):
    if f.endswith('.pt'):
        p = os.path.join(os.environ['LGBT_WEIGHTS_DIR'], f)
        print('   %-40s %6.1f MB' % (f, os.path.getsize(p) / 1e6))

df = pd.read_csv('predictions.csv')
print('\npredictions:', len(df))
print(df['stereotype_pred'].value_counts().to_dict())
display(df.head())

### If you are not in Colab

Everything above reduces to three commands:

```bash
git clone https://github.com/AmnO-O/Try.git
pip install -r requirements.txt
python src/train.py --epochs 10 --readout-init random
```

Put the TSVs in `./LGBT/` or export `LGBT_DATA_DIR`.